Name: Anwil Shien B. Oli

Section:BSCPE Ladderized

## CPE017 - Digital Signal Processing
This notebook contains discussion and activities for Module 4: Noise

## Noise
* means an unwanted or unpleasant sound
* In the context of signal processing, it has two different senses:
1. As in English, it can mean an unwanted signal of any kind. If two
signals interfere with each other, each signal would consider the other
to be noise.
2. “Noise” also refers to a signal that contains components at many frequencies, so it lacks the harmonic structure of the periodic signals.

The simplest noise to generate is uncorrelated uniform (UU) noise:

* “Uniform” means the signal contains random values from a uniform distribution; that is, every value in the range is equally likely.
* “Uncorrelated” means that the values are independent; that is, knowing one value provides no information about the others.

The following example generates UU noise with duration 0.5 seconds at 11,025 samples per second.

The output sounds like the static you hear if you tune a radio between channels.

In [ ]:
from thinkdsp import UncorrelatedUniformNoise, decorate

signal = UncorrelatedUniformNoise()   # Creates an instance of uncorrelated uniform noise
wave = signal.make_wave(duration=0.5, framerate=11025)  # Generates a waveform with a duration of 0.5 seconds and a sampling rate of 11,025 samples per second
wave.make_audio()   # Plays the audio corresponding to the generated waveform

Here's what a segment of it looks like:

In [ ]:
"A segment of the waveform is plotted, showing its amplitude variation over time."
segment = wave.segment(duration=0.1)
segment.plot()
decorate(xlabel='Time (s)',
         ylabel='Amplitude')

And here's the spectrum:

In [ ]:

spectrum = wave.make_spectrum()
spectrum.plot(linewidth=0.5)
decorate(xlabel='Frequency (Hz)',
         ylabel='Amplitude')

UU noise has the same power at all frequencies, on average, which we can confirm by looking at the normalized cumulative sum of power, called an integrated spectrum:

In [ ]:
integ = spectrum.make_integrated_spectrum()
integ.plot_power()
decorate(xlabel='Frequency (Hz)',
         ylabel='Cumulative power')

A straight line in this figure indicates that UU noise has equal power at all frequencies, on average.  By analogy with light, noise with this property is called "white noise".

### Brownian noise

Brownian noise is generated by adding up a sequence of random steps.

In [ ]:
from thinkdsp import BrownianNoise

signal = BrownianNoise()
wave = signal.make_wave(duration=0.5, framerate=11025)
wave.make_audio()

The sound is less bright, or more muffled, than white noise.

Here's what the wave looks like:

In [ ]:
wave.plot(linewidth=1)
decorate(xlabel='Time (s)',
         ylabel='Amplitude')

Here's what the power spectrum looks like on a linear scale.

In [ ]:
spectrum = wave.make_spectrum()
spectrum.plot_power(linewidth=0.5)
decorate(xlabel='Frequency (Hz)',
         ylabel='Power')

So much of the energy is at low frequencies, we can't even see the high frequencies.

### Pink noise

In [ ]:
from thinkdsp import PinkNoise

signal = PinkNoise(beta=0)
wave = signal.make_wave(duration=0.5)
wave.make_audio()

With $\beta=1$, pink noise has the relationship $P = K / f$, which is why it is also called $1/f$ noise.

In [ ]:
signal = PinkNoise(beta=1)
wave = signal.make_wave(duration=0.5)
wave.make_audio()

With $\beta=2$, we get Brownian (aka red) noise.

In [ ]:
signal = PinkNoise(beta=2)
wave = signal.make_wave(duration=0.5)
wave.make_audio()

Conclusion
The analysis of Bitcoin (BTC-USD) closing prices reveals that its power spectrum on a log-log scale typically exhibits a downward trend. This characteristic suggests that Bitcoin prices do not resemble white noise, which would have a flat power spectrum. Instead, the observed downward slope indicates a higher concentration of power at lower frequencies (long-term trends) compared to higher frequencies (short-term fluctuations).

This behavior is generally consistent with either pink noise (1/f noise) or Brownian noise (1/f² noise). Pink noise shows a power decrease proportional to 1/f (a slope of -1 on a log-log plot), while Brownian noise shows a decrease proportional to 1/f² (a slope of -2). Financial time series, including Bitcoin, often display properties closer to pink noise, characterized by long-range correlations and memory effects, meaning past values have a persistent influence on future values. Therefore, based on the spectral analysis, Bitcoin prices tend to behave more like pink noise or a signal with similar power-law decay characteristics, rather than completely random white noise.


### Assignment

1. At Yahoo Finance (https://finance.yahoo.com/quote/BTC-USD/) you can download the daily price of a BitCoin as a CSV file. Read this file and compute the spectrum of BitCoin prices as a function of time. Does it resemble white, pink, or Brownian noise?

### Bitcoin Price Analysis

In [ ]:
# Install yfinance to easily download historical data
!pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from thinkdsp import Wave, Spectrum, decorate

# Download Bitcoin (BTC-USD) historical data
btc_ticker = yf.Ticker("BTC-USD")
btc_data = btc_ticker.history(period="max")

# Display the first few rows of the data
display(btc_data.head())

#### Plotting Bitcoin Closing Price

Now that we have the Bitcoin price data, let's visualize the 'Close' prices over time to get a general understanding of its trend.

In [ ]:
# Plot the closing price
plt.figure(figsize=(14, 7))
btc_data['Close'].plot()
plt.title('Bitcoin (BTC-USD) Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.grid(True)
plt.show()

#### Preparing Data for Spectrum Analysis

To compute the spectrum, we need to treat the 'Close' price as a signal. Since `thinkdsp` expects a uniformly sampled signal, we'll use the daily closing prices as our signal data. We'll ensure uniform spacing and then create a `Wave` object, which is the necessary input for `thinkdsp`'s spectrum calculations.

In [ ]:
# Use 'Close' prices as our signal
signal_values = btc_data['Close'].values

# Create a Wave object. The framerate here represents the sampling rate (e.g., 1 sample per day).
# We'll use a nominal framerate for the purpose of spectrum analysis.
# A framerate of 1 means 1 sample per unit of time (e.g., day).
# It's important to understand that 'time' here is in days.

# We need to handle potential NaN values or ensure the signal is continuous
# For simplicity, let's drop NaNs if any for the signal processing part
clean_signal = btc_data['Close'].dropna()

# Assuming daily sampling, our framerate is 1 sample per day
# The duration will be the number of days in the clean_signal
# If we consider 'time' in days, then framerate = 1 (sample/day)
# The number of samples is len(clean_signal)

# The thinkdsp library requires a `Wave` object, which typically represents a time-series signal.
# The 'framerate' defines the number of samples per unit of time. If our data is daily,
# then a framerate of 1 makes sense, where each unit of time is 1 day.
# However, for FFT, the interpretation of frequency will depend on this framerate.
# Let's consider 1 sample per day, so our 'frequency' will be in cycles per day.
wave_btc = Wave(clean_signal.values, framerate=1)

# Compute the spectrum
spectrum_btc = wave_btc.make_spectrum()

# Plot the power spectrum (on a log-log scale is often useful for noise analysis)
spectrum_btc.plot_power(yscale='log', xscale='log', linewidth=0.7)
decorate(xlabel='Frequency (cycles/day)', ylabel='Power')
plt.title('Power Spectrum of Bitcoin Closing Prices')
plt.grid(True, which="both", ls="-")
plt.show()

#### Plotting the Integrated Spectrum

Next, let's examine the integrated spectrum. This plot helps in comparing the observed noise type with theoretical white, pink, or Brownian noise, providing a clearer view of cumulative power across frequencies.

In [ ]:
# Compute the integrated spectrum
integ_btc = spectrum_btc.make_integrated_spectrum()

# Plot the integrated spectrum on a log-log scale for better comparison
integ_btc.plot_power(xscale='log', yscale='log')
decorate(xlabel='Frequency (cycles/day)',
         ylabel='Cumulative power (log scale)')
plt.title('Integrated Power Spectrum of Bitcoin Closing Prices')
plt.grid(True, which="both", ls="-")
plt.show()

### Interpretation of the Spectrum:

*   **White Noise:** The power spectrum is flat (equal power at all frequencies). The integrated spectrum would show a linear increase on a linear plot, or a curve on a log-log plot.
*   **Pink Noise (1/f noise):** The power decreases as `1/f` (or `f^-1`). On a log-log plot, the power spectrum would show a straight line with a slope of -1. The integrated spectrum would also show a curve on a log-log plot, but with a different characteristic.
*   **Brownian Noise (Red Noise / 1/f^2 noise):** The power decreases as `1/f^2` (or `f^-2`). On a log-log plot, the power spectrum would show a straight line with a slope of -2. The integrated spectrum would tend to accumulate more power at lower frequencies.

By observing the slope of the power spectrum on a log-log plot, we can infer whether the Bitcoin price series resembles white, pink, or Brownian noise. If the slope is approximately -1, it's pink noise. If it's approximately -2, it's Brownian noise. If it's flat (slope 0), it's white noise.

From the plots above, especially the power spectrum on a log-log scale, you should observe a general downward trend. The slope of this trend indicates the `beta` value. Bitcoin prices often exhibit characteristics closer to **Pink Noise** or even **Brownian Noise** at lower frequencies, meaning there's more power in the lower frequencies (long-term trends) and less in the higher frequencies (short-term fluctuations), unlike white noise which has uniform power across all frequencies. Specifically, a `1/f` relationship (pink noise) is often observed in financial time series.

White Noise Implementation

This Processing sketch creates a simple white noise generator using the Processing Sound library. White noise is characterized by having equal power across all frequencies, resulting in a consistent, random sound that can be harsh to the ear, especially at high frequencies.

// This line imports the Sound library in Processing, which provides various tools for sound generation and manipulation. In this case, it's used to create and control white noise.

`import processing.sound.*;`

// This declares an object named noise of type WhiteNoise. The WhiteNoise class from the Sound library allows us to generate random noise that sounds similar to static or the hiss of an untuned radio.

`WhiteNoise noise;`

// The setup() function is called once at the start.
size(640, 360) sets the window size to 640 pixels wide and 360 pixels tall.
background(255) sets the background color to white (255 is the maximum value for a grayscale color in Processing).

`void setup() {`

`  size(640, 360);`

`  background(255);`

// noise = new WhiteNoise(this) creates a new WhiteNoise object, passing the Processing sketch (this) as an argument.
noise.play() starts the white noise sound. This will continuously play the noise until the program ends or noise.stop() is called.

`noise = new WhiteNoise(this);`

`  noise.play();`

`}`

// draw() is a loop that runs continuously while the program is running.

// map(mouseX, 0, width, -1.0, 1.0) is used to map the current horizontal position of the mouse (mouseX) to a value between -1.0 (left speaker) and 1.0 (right speaker). The noise.pan() function adjusts the "panning," or the left-right position of the sound, based on the mouse position.

// If the mouse is on the left side of the window, the sound will come more from the left speaker.
If the mouse is on the right side of the window, the sound will come more from the right speaker.

`void draw() {`

`  // Map mouseX from -1.0 to 1.0 for left to right`

`  noise.pan(map(mouseX, 0, width, -1.0, 1.0));`

// map(mouseY, 0, height, 0.3, 0.0) maps the current vertical position of the mouse (mouseY) to a value between 0.3 and 0.0.
// When the mouse is at the top of the window (mouseY = 0), the amplitude (volume) will be 0.3 (loud).
// When the mouse is at the bottom of the window (mouseY = height), the amplitude will be 0.0 (silent).
// noise.amp() adjusts the amplitude (loudness) of the white noise based on the mouse's vertical position, allowing dynamic control of volume.

`// Map mouseY from 0.0 to 0.3 for amplitude`

`  // (the higher the mouse position, the louder the sound)`

`  noise.amp(map(mouseY, 0, height, 0.3, 0.0));`

`}`

### Full Code

/**
 * This is a simple white noise generator. White noise has equal power at all
 * frequencies. The high frequencies can make it very grating to the ear.
 */

import processing.sound.*;

WhiteNoise noise;

void setup() {
  size(640, 360);
  background(255);

  // Create and start the noise generator
  noise = new WhiteNoise(this);
  noise.play();
}      

void draw() {
  // Map mouseX from -1.0 to 1.0 for left to right
  noise.pan(map(mouseX, 0, width, -1.0, 1.0));

  // Map mouseY from 0.0 to 0.3 for amplitude
  // (the higher the mouse position, the louder the sound)
  noise.amp(map(mouseY, 0, height, 0.3, 0.0));
}

### Brown Noise

/**
 * This is a simple "Brown" (also known as "Brownian" or "red") noise generator.
 * Its power decreases by 6dB per octave, giving it a much "softer" quality than
 * white or pink noise.
 */

import processing.sound.*;

BrownNoise noise;

void setup() {
  size(640, 360);
  background(255);

  // Create and start the noise generator
  noise = new BrownNoise(this);
  noise.play();
}      

void draw() {
  // Map mouseX from -1.0 to 1.0 for left to right
  noise.pan(map(mouseX, 0, width, -1.0, 1.0));

  // Map mouseY from 0.0 to 0.3 for amplitude
  // (the higher the mouse position, the louder the sound)
  noise.amp(map(mouseY, 0, height, 0.3, 0.0));
}

### Pink Noise

/**
 * This is a simple pink noise generator. The energy of pink noise falls off at 3 dB
 * per octave, which puts it somewhere between White and Brownian noise.
 */

import processing.sound.*;

PinkNoise noise;

void setup() {
  size(640, 360);
  background(255);

  // Create and start noise generator
  noise = new PinkNoise(this);
  noise.play();
}      

void draw() {
  // Map mouseX from -1.0 to 1.0 for left to right
  noise.pan(map(mouseX, 0, width, -1.0, 1.0));

  // Map mouseY from 0.0 to 0.5 for amplitude
  // (the higher the mouse position, the louder the sound)
  noise.amp(map(mouseY, 0, height, 0.5, 0.0));
}